# 01 — Mean-Variance: GMV & Tangency

The Markowitz core: the global minimum-variance portfolio and the
maximum-Sharpe (tangency) portfolio, the two long-only strategies every later
method in this repo is measured against.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt

import maplab as ml
import _nb01_helpers as h
from maplab import Panel, apply_style
from maplab.models import tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")

## The math

**Unconstrained global minimum-variance (GMV):**

$$w_{\text{GMV}} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}'\Sigma^{-1}\mathbf{1}}$$

**Unconstrained tangency (maximum Sharpe):**

$$w_{\text{tan}} = \frac{\Sigma^{-1}(\mu - r_f)}{\mathbf{1}'\Sigma^{-1}(\mu - r_f)}$$

**Two-fund separation:** every unconstrained mean-variance-efficient portfolio
is a combination of just these two funds, $w_{GMV}$ and $w_{tan}$ — no other
information from $\mu$ or $\Sigma$ is needed once you have them.

**Why the classes in `maplab.models` are long-only, not these closed forms:**
the unconstrained solutions above can (and, on this universe, do) put large
long and short positions on assets with noisy estimated means — leverage and
short exposure that a real long-only allocator can't take and that is mostly
estimation error, not signal. `GMV` and `MaxSharpe` solve the same objectives
numerically with $w \ge 0$ added, which is why they need `scipy.optimize`
rather than a one-line solve.

**$\mu$ estimation (log → arithmetic):** the harness estimates $(\mu, \Sigma)$
on **log** returns (additive, well-behaved) but scores strategies on
**arithmetic** (simple) returns. Feeding a log-return mean straight into a
mean-variance objective is systematically biased downward, so:

$$\mu = \overline{r_{\log}} \times 252 + \tfrac{1}{2}\,\text{diag}(\Sigma)$$

**Sharpe:**

$$\text{Sharpe} = \frac{\text{mean}(r - r_f)}{\text{std}(r - r_f)} \times \sqrt{252}$$

where $r_f$ is BIL's daily simple return (`ml.load_rf_returns()`) — the
actual T-bill carry, not a fixed hurdle. `MaxSharpe` uses the trailing-252-day
mean of that same BIL series, annualized, as $r_f$ at each rebalance
(`rf="panel"`, the default — see `MaxSharpe._rf_ann`).

## Snapshot at the train/test split

$(\mu, \Sigma)$ estimated at the last rebalance on or before
`TRAIN_TEST_SPLIT`, and what GMV, MaxSharpe, and the **unconstrained**
tangency portfolio each do with them. The unconstrained column is the
"why long-only" evidence: its largest longs and shorts, and its gross
leverage $\sum |w|$.

In [ ]:
from maplab import GMV, MaxSharpe

rdates = ml.rebalance_dates(simple_returns.index)
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
split_rebal = rdates[rdates <= split_ts].max()
print(f"Snapshot date (last rebalance <= {ml.TRAIN_TEST_SPLIT}): {split_rebal.date()}")

gmv_snap = GMV()
mu, Sigma = gmv_snap._estimate(panel, split_rebal)

rf_ann = MaxSharpe()._rf_ann(panel, split_rebal)
print(f"rf_ann at snapshot (trailing-252-day BIL mean, annualized): {rf_ann:.4%}")

w_gmv = GMV()(panel, split_rebal)
w_msr = MaxSharpe()(panel, split_rebal)

try:
    w_tan = tangency_closed_form(mu, Sigma, rf=rf_ann)
    tan_undefined = False
except ValueError as e:
    tan_undefined = True
    tan_error = str(e)

cols = {"GMV (long-only)": w_gmv, "MaxSharpe (long-only)": w_msr}
if not tan_undefined:
    cols["Tangency (UNCONSTRAINED)"] = w_tan
snap = pd.DataFrame(cols).round(4)

leverage_msg = (f"Gross leverage sum|w|  —  GMV: {w_gmv.abs().sum():.3f}  "
                f"MaxSharpe: {w_msr.abs().sum():.3f}")
if not tan_undefined:
    leverage_msg += f"  Unconstrained tangency: {w_tan.abs().sum():.3f}"
print(leverage_msg)

if tan_undefined:
    print(f"\nUnconstrained tangency is UNDEFINED at this snapshot:\n  {tan_error}")
    print("(1'Σ⁻¹(μ−rf) = (1'Σ⁻¹1)(μ_GMV,unc − rf), and 1'Σ⁻¹1 > 0, so the "
          "denominator is negative iff the unconstrained GMV's own expected "
          "return is below rf — the tangency then lies on the inefficient "
          "branch of the hyperbola, not a numerically \"extreme\" point on "
          "the efficient one.)")
else:
    print("\nUnconstrained tangency — largest LONGS:")
    display(w_tan.sort_values(ascending=False).head(5).round(4).to_frame("weight"))
    print("Unconstrained tangency — largest SHORTS:")
    display(w_tan.sort_values().head(5).round(4).to_frame("weight"))

print("\nHoldings with any weight > 1e-4:")
mask = (snap.abs() > 1e-4).any(axis=1)
snap.loc[mask].sort_values("GMV (long-only)", ascending=False)

In [ ]:
from maplab.models import gmv_closed_form

w_unc = gmv_closed_form(Sigma)
mu_gmv_unc = float(w_unc.to_numpy() @ mu.to_numpy())
ones_Sinv_ones = float(np.ones(len(mu)) @ np.linalg.solve(Sigma.to_numpy(), np.ones(len(mu))))
denom_from_identity = ones_Sinv_ones * (mu_gmv_unc - rf_ann)
raw_denom = float(np.ones(len(mu)) @ np.linalg.solve(Sigma.to_numpy(), (mu - rf_ann).to_numpy()))

print(f"mu' w_unc (unconstrained GMV's own expected return): {mu_gmv_unc:.6f}  (< rf_ann={rf_ann:.6f}? {mu_gmv_unc < rf_ann})")
print(f"1'Σ⁻¹1: {ones_Sinv_ones:.6f}")
print(f"(1'Σ⁻¹1) * (mu_GMV,unc - rf_ann): {denom_from_identity:.6f}   vs.   raw denominator 1'Σ⁻¹(μ-rf_ann): {raw_denom:.6f}")

print(f"\nUnconstrained GMV gross leverage sum|w|: {w_unc.abs().sum():.3f}")
print("\nUnconstrained GMV — largest LONGS:")
display(w_unc.sort_values(ascending=False).head(5).round(4).to_frame("weight"))
print("Unconstrained GMV — largest SHORTS:")
display(w_unc.sort_values().head(5).round(4).to_frame("weight"))

What does the long-only constraint remove at 2022-12-31? Unconstrained GMV and tangency (closed forms) vs the long-only solutions used in the backtest; the table gives gross leverage and total short weight. The tangency is where estimation error in μ is amplified most — the reason MaxSharpe is run long-only and still corner-hops.

In [ ]:
fig, leverage_table = h.unconstrained_vs_long_only(Sigma, mu, rf_ann, w_gmv, w_msr)
plt.show()

print(leverage_table.to_string(float_format=lambda v: f"{v:.4f}"))

## The long-only efficient frontier

At the same snapshot date: sweep a target return, minimize variance subject
to $w'\mu = \text{target}$, $\sum w = 1$, $w \ge 0$ (SLSQP with analytic
gradients). The 13 assets are shown as labeled dots, GMV and MaxSharpe are
marked, and the capital market line (CML) starts at $(0, r_f)$ — the
snapshot's trailing-252-day BIL mean — and passes through MaxSharpe. $\mu$
here is the trailing-252-day arithmetic mean, so the tangency tilts hard
toward last year's winners — at end-2022, ~74% in the dollar (UUP).

The grey backdrop is 20,000 random long-only portfolios drawn from a
Dirichlet($\alpha$) distribution on the 13 assets. Dirichlet(1) — the
textbook choice on 6 assets (median effective N $\approx$ 3.7) — spreads out
on 13 assets (median effective N $\approx$ 7.3, median max weight $\approx$
23%) and never reaches the corners where GMV and MaxSharpe live. The next
cell sweeps $\alpha \in \{1, 0.3, 0.1\}$ on the 13-asset universe and picks
the one whose median effective N is closest to that 6-asset reference:
$\alpha=0.3$ (median effective N $\approx$ 4.1, median max weight $\approx$
38%).

In [ ]:
n = len(ml.UNIVERSE)
reference_eff_n = h.effective_n_stats(6, 1.0)["median_eff_n"]  # Dirichlet(1) on 6 assets, the textbook case

dirichlet_table = pd.DataFrame([h.effective_n_stats(n, a) for a in (1.0, 0.3, 0.1)]).set_index("alpha")
dirichlet_table["abs_diff_from_ref"] = (dirichlet_table["median_eff_n"] - reference_eff_n).abs()
mc_alpha = float(dirichlet_table["abs_diff_from_ref"].idxmin())

print(f"Reference: Dirichlet(1) on 6 assets — median effective N = {reference_eff_n:.3f}")
print(f"Chosen alpha on {n} assets: {mc_alpha} "
      f"(median effective N {dirichlet_table.loc[mc_alpha, 'median_eff_n']:.3f}, "
      f"median max weight {dirichlet_table.loc[mc_alpha, 'median_max_w']:.3f})")
dirichlet_table.round(4)

In [ ]:
fig = h.frontier_figure(Sigma, mu, w_gmv, w_msr, mc_alpha, rf_ann, split_rebal)
plt.show()

## Backtest: GMV, MaxSharpe, EqualWeight

Fresh strategy instances run through `ml.backtest` (default scoring start —
the first eligible rebalance after the 252-day warm-up). `ml.summary` over
the full sample, the train window ($\le$ `TRAIN_TEST_SPLIT`), and the held-out
test window (> split). Wealth and drawdown figures move to the Phase 2 comparison notebook.

In [ ]:
from maplab import EqualWeight, backtest, summary

strategies = {"GMV": GMV(), "MaxSharpe": MaxSharpe(), "EqualWeight": EqualWeight()}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag}

split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    windows = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in windows.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

msr = strategies["MaxSharpe"]
print(f"MaxSharpe fallback_dates: {len(msr.fallback_dates)}   retry_dates: {len(msr.retry_dates)}")

summary_table.round(4)

## Allocation through time

Target weights from each strategy's `wlog`, aggregated by `ml.GROUP_OF` into
the 6 asset groups, at every rebalance.

In [ ]:
gmv_group = ml.diagnostics.weights_by_group(results["GMV"]["wlog"])
msr_group = ml.diagnostics.weights_by_group(results["MaxSharpe"]["wlog"])

fig = ml.plotting.group_stackplot(results, ["GMV", "MaxSharpe"], figsize=(9, 8),
                                  title_fmt="{name} — target weights by asset group", split_ts=split_ts)
plt.show()

print("Mean target weight by asset group:")
group_means = pd.DataFrame({"GMV": gmv_group.mean(), "MaxSharpe": msr_group.mean()})
display(group_means.round(4))

uup_w = results["GMV"]["wlog"]["UUP"]
print(f"GMV: UUP mean weight = {uup_w.mean():.4f}  (min={uup_w.min():.4f}, max={uup_w.max():.4f})")

How concentrated are GMV and MaxSharpe over time, and how much do they trade? MaxSharpe's corner-hopping shows up as a median max weight of 44% (up to 100%) and high turnover. Rows: effective N, max weight, UUP weight, trailing 12-rebalance one-way turnover (%/yr; initial allocation excluded); EW dashed.

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["GMV", "MaxSharpe"],
    rows=("effN", "max_w", "highlight", "turnover"), ref="EqualWeight", split_ts=split_ts,
    figsize=(9.5, 14), colors={name: ml.FAMILY_COLORS[strategies[name].family] for name in ["GMV", "MaxSharpe"]},
    suptitle="GMV vs MaxSharpe: effN, max weight, UUP weight, turnover",
)
plt.show()

Capital vs risk at 2022-12-31 (C5): bars = capital weight, faded bars = risk-contribution share, dashed = 1/13; a negative risk share means the asset hedges the rest of the portfolio.

In [ ]:
weights_c5 = {"GMV": results["GMV"]["wlog"].loc[split_rebal], "MaxSharpe": results["MaxSharpe"]["wlog"].loc[split_rebal],
              "EW": results["EqualWeight"]["wlog"].loc[split_rebal]}
print(ml.diagnostics.diversification_table(weights_c5, Sigma).to_string(float_format=lambda v: f"{v:.4f}"))

fig = ml.plotting.capital_vs_risk(weights_c5, Sigma, title=f"Capital weight vs risk share, {split_rebal.date()} (sample Σ)")
plt.show()

**TAKEAWAY**
- **On a clean, one-premium-per-asset universe the verdict is not "mean-variance loses."** Sharpe vs BIL: EW 0.74 ≈ MaxSharpe 0.72 > GMV 0.62 (full window). The ranking flips by window: EW leads pre-2023 (0.70 vs 0.60 / 0.59), MaxSharpe leads 2023+ (1.13 vs 0.91 / 0.76). That window-dependence is the repo's thesis in miniature.
- **None of those Sharpe gaps is statistically meaningful.** The i.i.d. standard error of a Sharpe ratio is ≈ √((1 + SR²/2)/T): about 0.27 over the full 17 years, about 0.7 over the 3.3-year test window.
- **What *is* robust is the risk profile and what it costs to get there.** GMV: 3.2% vol, −6.7% max DD. MaxSharpe: 6.3% vol, −10.8%. EW: 7.9% vol, −17.8%. MaxSharpe turns over ~248%/yr, 18× EW's 14%, and its weights jump to near-100% single-group corners (equity 2014 and 2017–18, TIPS 2021, 74% dollar at end-2022). The error-maximizer shows up in the weights and the trading, not in the Sharpe ratio.
- **GMV no longer hides in cash.** With BIL as the numeraire it must choose among risk premia, and it leans on the dollar (UUP 37% on average, range 18–58%) because the dollar rallies when risk assets sell off: a diversifier, not a low-vol accident.
- **Long-only is not cosmetic.** At end-2022 the unconstrained tangency is defined but carries 13× gross leverage (+193% UUP, −216% TIPS).
- **Estimated vs realized efficiency:** ex-ante, EW sits deep inside the frontier cloud; ex-post it has the highest wealth. Sample-mean μ is the weak link. Better μ (03, Black-Litterman) and better Σ (04, shrinkage) come next. One historical path, no bootstrap confidence intervals yet (Phase 2).